# Notebook 05 — Temporal Fusion Transformer (Simplified)

**Goal:** Train a CPU-friendly TFT — the state-of-the-art for interpretable time-series forecasting.

Key TFT innovations: variable selection (GRN), multi-head attention, gated skip connections.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.preprocessing import StandardScaler

from src.utils import set_seed, evaluate_classifier, plot_confusion_matrix
from src.models import SimpleTFT, LOBDataset
from src.data_loader import create_sequences, train_val_test_split

set_seed(42)
pl.seed_everything(42)
%matplotlib inline

RESULTS = Path('../results')
SEQ_LEN = 50
BATCH_SIZE = 256
MAX_EPOCHS = 15
print(f'TFT — CPU mode | Seq: {SEQ_LEN} | Batch: {BATCH_SIZE}')

## 1. Load Sequence Data

In [ ]:
df = pd.read_parquet('../data/processed/features.parquet')
feature_cols = [c for c in df.columns if not c.startswith('label')]

X = np.nan_to_num(df[feature_cols].values.astype(np.float32))
y = df['label'].values.astype(np.int64)

scaler = StandardScaler()
n_train = int(len(X) * 0.7)
scaler.fit(X[:n_train])
X_scaled = scaler.transform(X)

X_seq, y_seq = create_sequences(X_scaled, y, seq_len=SEQ_LEN)
splits = train_val_test_split(X_seq, y_seq, train_ratio=0.7, val_ratio=0.15)

train_dl = DataLoader(LOBDataset(splits['X_train'], splits['y_train']),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_dl = DataLoader(LOBDataset(splits['X_val'], splits['y_val']),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl = DataLoader(LOBDataset(splits['X_test'], splits['y_test']),
                     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

n_features = X_seq.shape[2]
print(f'n_features: {n_features}')

## 2. Train Simplified TFT

In [ ]:
tft_model = SimpleTFT(
    n_features=n_features, hidden_dim=32,
    n_heads=4, n_classes=3, dropout=0.2, lr=1e-3
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator='cpu',
    enable_progress_bar=True,
    callbacks=[pl.callbacks.EarlyStopping('val_loss', patience=5)],
    logger=False,
)

trainer.fit(tft_model, train_dl, val_dl)
print('TFT training complete.')

## 3. Evaluate TFT

In [ ]:
tft_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_batch, y_batch in test_dl:
        logits = tft_model(x_batch)
        all_preds.append(logits.argmax(1).numpy())
        all_labels.append(y_batch.numpy())

tft_preds = np.concatenate(all_preds)
tft_labels = np.concatenate(all_labels)

tft_metrics = evaluate_classifier(tft_labels, tft_preds, title='TFT (Simplified)')
plot_confusion_matrix(tft_labels, tft_preds, title='Temporal Fusion Transformer',
                     save_path=str(RESULTS / 'plots' / 'cm_tft.png'))

torch.save(tft_model.state_dict(), RESULTS / 'models' / 'tft.pt')
np.save('../data/processed/tft_preds.npy', tft_preds)
print('TFT predictions saved.')

## 4. All Models Comparison

In [ ]:
# Load baseline results
baselines = pd.read_csv(RESULTS / 'tables' / 'baseline_results.csv')
lstm_gru = pd.read_csv(RESULTS / 'tables' / 'lstm_gru_results.csv')

tft_row = pd.DataFrame([{'Model': 'TFT (Simplified)', **tft_metrics}])
all_results = pd.concat([baselines, lstm_gru, tft_row], ignore_index=True)
all_results.to_csv(RESULTS / 'tables' / 'all_model_results.csv', index=False)

print('\n=== All Model Results ===')
print(all_results.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ['accuracy', 'f1_macro', 'f1_weighted']):
    ax.barh(all_results['Model'], all_results[metric], color='#3498db')
    ax.set_xlabel(metric.replace('_', ' ').title())
    ax.set_xlim(0, 1)
    for i, v in enumerate(all_results[metric]):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)

fig.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
fig.savefig(RESULTS / 'plots' / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- TFT adds variable selection (decides which features matter) and multi-head attention
- CPU-friendly simplified version trains in reasonable time
- Comparison table shows relative strengths of each approach

**Next:** Notebook 06 — Backtesting